In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.datasets import make_classification

Se tienen los dataframes para distintos porcentajes de etiquetas en los datos (5%, 10%, 20%)

In [9]:
df5train = pd.read_csv('train_5_porc.csv')
df5train

,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Survived
0,1.190931,-0.633681,-0.497692,0.382045,0,0,1,0,0,0
1,-1.156464,3.249031,2.098923,0.478736,0,1,1,0,1,0
2,-0.080013,-0.633681,-0.497692,-0.859250,0,1,1,0,1,0
3,-0.014386,-0.633681,-0.497692,-0.353367,1,0,0,0,1,1
4,0.579884,1.038512,-0.497692,1.548433,0,0,0,1,0,1
...,...,...,...,...,...,...,...,...,...,...
329,0.048854,-0.633681,-0.497692,0.986020,0,0,1,0,1,-1
330,-0.370504,-0.633681,1.140590,0.078301,1,0,0,0,1,-1
331,0.753919,-0.633681,-0.497692,-0.864595,0,1,1,0,1,-1
332,-0.014386,1.038512,1.140590,0.125453,0,1,0,0,1,-1


In [4]:
df10train = pd.read_csv('train_10_porc.csv')
df10train.head()

,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Survived
0,-0.148214,-0.633681,-0.497692,-0.353367,1,0,1,0,1,0
1,0.048854,-0.633681,-0.497692,-0.876361,0,1,0,1,0,1
2,1.190931,-0.633681,-0.497692,0.344365,0,0,0,0,0,1
3,0.048854,-0.633681,-0.497692,-0.939901,0,1,1,0,0,0
4,-0.080013,-0.633681,-0.497692,-0.859250,0,1,1,0,1,0


In [5]:
df20train = pd.read_csv('train_20_porc.csv')
df20train.head()

,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Survived
0,-0.219200,-0.633681,-0.497692,-0.870955,0,1,1,0,1,0
1,-0.719452,-0.633681,-0.497692,0.888262,0,0,1,0,1,0
2,-0.625434,-0.633681,-0.497692,-0.593383,1,0,1,0,1,0
3,-0.014386,-0.633681,-0.497692,-0.353367,1,0,0,0,1,1
4,0.048854,1.038512,4.944562,1.284875,0,1,0,0,1,1


In [10]:
#Dataframe de prueba
dftest = pd.read_csv('test.csv')
dftest

,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S,Survived
0,0.334561,-0.633681,-0.497692,-0.389701,1,0,1,0,1,0
1,0.048854,1.038512,-0.497692,-1.044639,0,1,1,0,0,0
2,0.485866,-0.633681,-0.497692,-0.438927,1,0,1,1,0,0
3,-1.156464,-0.633681,2.098923,0.042386,0,1,1,0,1,0
4,1.513330,1.038512,-0.497692,1.365324,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...
79,0.048854,-0.633681,-0.497692,-0.859250,0,1,1,0,1,0
80,-0.451398,-0.633681,-0.497692,-0.327799,1,0,1,0,0,0
81,0.109874,-0.633681,-0.497692,0.290353,1,0,1,0,1,0
82,0.986118,-0.633681,-0.497692,-0.593383,1,0,1,0,1,0


Separación de datos:

Se sigue el proceso hecho en el archivo de EDA

In [23]:
df = pd.read_csv('titanic.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,0,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,1,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,0,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,0,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,1,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [24]:
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

In [25]:
#Tratamiento de nulos
mediana_age = df['Age'].median()
df['Age'] = df['Age'].fillna(mediana_age)
mediana_fare = df['Fare'].median()
df['Fare'] = df['Fare'].fillna(mediana_fare)

In [26]:
#Se aplica una escala logarítmica
columnas = ['Age', 'SibSp', 'Parch', 'Fare']
for columna in columnas:
    df[columna] = np.log1p(df[columna])

In [27]:
scaler = StandardScaler()
df[columnas] = scaler.fit_transform(df[columnas])
columnas_categoricas = ['Pclass', 'Sex', 'Embarked']

df = pd.get_dummies(df, columns=columnas_categoricas, drop_first=True, dtype=int)

In [28]:
df.head()

,Survived,Age,SibSp,Parch,Fare,Pclass_2,Pclass_3,Sex_male,Embarked_Q,Embarked_S
0,0,0.461545,-0.633681,-0.497692,-0.867031,0,1,1,1,0
1,1,0.986118,1.038512,-0.497692,-0.969149,0,1,0,0,1
2,0,1.458985,-0.633681,-0.497692,-0.669252,1,0,1,1,0
3,0,0.048854,-0.633681,-0.497692,-0.773647,0,1,1,0,1
4,1,-0.293207,1.038512,1.140590,-0.443786,0,1,0,0,1


In [29]:
target = 'Survived'
X = df.drop(columns=[target])
y = df[target]

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)

Transductive SVM 

Para el 5%

In [33]:
y5train=df5train['Survived']
base_model5 = SVC(probability=True)

modelo1 = SelfTrainingClassifier(base_model5)

modelo1.fit(X_train, y5train)

,"estimator estimator: estimator objectAn estimator object implementing `fit` and `predict_proba`.Invoking the `fit` method will fit a clone of the passed estimator,which will be stored in the `estimator_` attribute... versionadded:: 1.6 `estimator` was added to replace `base_estimator`.",SVC(probability=True)
,"threshold threshold: float, default=0.75The decision threshold for use with `criterion='threshold'`.Should be in [0, 1). When using the `'threshold'` criterion, a:ref:`well calibrated classifier ` should be used.",0.75
,"criterion criterion: {'threshold', 'k_best'}, default='threshold'The selection criterion used to select which labels to add to thetraining set. If `'threshold'`, pseudo-labels with predictionprobabilities above `threshold` are added to the dataset. If `'k_best'`,the `k_best` pseudo-labels with highest prediction probabilities areadded to the dataset. When using the 'threshold' criterion, a:ref:`well calibrated classifier ` should be used.",'threshold'
,"k_best k_best: int, default=10The amount of samples to add in each iteration. Only used when`criterion='k_best'`.",10
,"max_iter max_iter: int or None, default=10Maximum number of iterations allowed. Should be greater than or equalto 0. If it is `None`, the classifier will continue to predict labelsuntil no new pseudo-labels are added, or all unlabeled samples havebeen labeled.",10
,"verbose verbose: bool, default=FalseEnable verbose output.",False
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0


Para el 10%

In [34]:
y10train=df10train['Survived']
base_model10 = SVC(probability=True)

modelo2 = SelfTrainingClassifier(base_model10)

modelo2.fit(X_train, y10train)

,"estimator estimator: estimator objectAn estimator object implementing `fit` and `predict_proba`.Invoking the `fit` method will fit a clone of the passed estimator,which will be stored in the `estimator_` attribute... versionadded:: 1.6 `estimator` was added to replace `base_estimator`.",SVC(probability=True)
,"threshold threshold: float, default=0.75The decision threshold for use with `criterion='threshold'`.Should be in [0, 1). When using the `'threshold'` criterion, a:ref:`well calibrated classifier ` should be used.",0.75
,"criterion criterion: {'threshold', 'k_best'}, default='threshold'The selection criterion used to select which labels to add to thetraining set. If `'threshold'`, pseudo-labels with predictionprobabilities above `threshold` are added to the dataset. If `'k_best'`,the `k_best` pseudo-labels with highest prediction probabilities areadded to the dataset. When using the 'threshold' criterion, a:ref:`well calibrated classifier ` should be used.",'threshold'
,"k_best k_best: int, default=10The amount of samples to add in each iteration. Only used when`criterion='k_best'`.",10
,"max_iter max_iter: int or None, default=10Maximum number of iterations allowed. Should be greater than or equalto 0. If it is `None`, the classifier will continue to predict labelsuntil no new pseudo-labels are added, or all unlabeled samples havebeen labeled.",10
,"verbose verbose: bool, default=FalseEnable verbose output.",False
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0


Para el 20%

In [35]:
y20train=df20train['Survived']
base_model20 = SVC(probability=True)

modelo3 = SelfTrainingClassifier(base_model20)

modelo3.fit(X_train, y20train)

,"estimator estimator: estimator objectAn estimator object implementing `fit` and `predict_proba`.Invoking the `fit` method will fit a clone of the passed estimator,which will be stored in the `estimator_` attribute... versionadded:: 1.6 `estimator` was added to replace `base_estimator`.",SVC(probability=True)
,"threshold threshold: float, default=0.75The decision threshold for use with `criterion='threshold'`.Should be in [0, 1). When using the `'threshold'` criterion, a:ref:`well calibrated classifier ` should be used.",0.75
,"criterion criterion: {'threshold', 'k_best'}, default='threshold'The selection criterion used to select which labels to add to thetraining set. If `'threshold'`, pseudo-labels with predictionprobabilities above `threshold` are added to the dataset. If `'k_best'`,the `k_best` pseudo-labels with highest prediction probabilities areadded to the dataset. When using the 'threshold' criterion, a:ref:`well calibrated classifier ` should be used.",'threshold'
,"k_best k_best: int, default=10The amount of samples to add in each iteration. Only used when`criterion='k_best'`.",10
,"max_iter max_iter: int or None, default=10Maximum number of iterations allowed. Should be greater than or equalto 0. If it is `None`, the classifier will continue to predict labelsuntil no new pseudo-labels are added, or all unlabeled samples havebeen labeled.",10
,"verbose verbose: bool, default=FalseEnable verbose output.",False
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
